# Healthcare and Medicine: AI-Powered Disease Prediction and Medical Information Assistant

## Project objective

This notebook demonstrates a complete machine-learning workflow for educational and research purposes:

1. Load multiple healthcare-related datasets independently.
2. Inspect and clean each dataset.
3. Check compatibility before considering any merge.
4. Select a suitable binary classification task.
5. Build leakage-resistant preprocessing pipelines.
6. Train and compare five classification algorithms.
7. Evaluate accuracy, precision, recall, F1-score, confusion matrices, and classification reports.
8. Export the complete pipeline to `disease_model.pkl`.
9. Reload the pipeline and generate an illustrative prediction.

> **Important:** This notebook is not a medical diagnosis system. Predictions must not be used for medical decisions. The datasets may be synthetic or otherwise non-clinical; their original documentation must be checked before making claims about provenance.


## 1. Project Introduction

The supplied repository contains a general disease-symptom dataset, a heart-disease dataset, and a diabetes-related dataset. No verified lung-disease target dataset was supplied in the request. Therefore, the notebook loads all three datasets for inspection and uses the heart-disease dataset as the default binary-classification demonstration when it is available. This prevents incorrectly presenting a heart-disease model as a lung-disease model.

## 2. Importing Libraries

In [1]:
# Install packages if needed in Google Colab
# !pip -q install pandas numpy scikit-learn seaborn matplotlib joblib requests

import io
import os
import re
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, Markdown

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
MODEL_FILE = "disease_model.pkl"
TEST_SIZE = 0.20


## 3. Dataset Sources

In [2]:
DATASET_SOURCES = {
    "general_disease_symptoms": {
        "name": "General Disease-Symptom Dataset",
        "url": "https://raw.githubusercontent.com/vyasanbmathew2008/Team-6/refs/heads/main/healthcare-disease-prediction/dataset/healthcare_dataset.csv",
        "target": "Disease",
        "description": "Disease labels with up to twelve symptom columns."
    },
    "heart_disease": {
        "name": "Heart Disease Dataset",
        "url": "https://raw.githubusercontent.com/vyasanbmathew2008/Team-6/refs/heads/main/heart_disease.csv",
        "target": "Heart Disease Status",
        "description": "Demographic, lifestyle, laboratory-style, and risk-factor columns."
    },
    "diabetes": {
        "name": "Diabetes Dataset",
        "url": "https://raw.githubusercontent.com/vyasanbmathew2008/Team-6/refs/heads/main/diabetes_dataset.csv",
        "target": "Target",
        "description": "Diabetes-related labels and many categorical and numerical attributes."
    }
}

sources_table = pd.DataFrame([
    {
        "dataset_key": key,
        "dataset_name": value["name"],
        "target_column": value["target"],
        "source_url": value["url"]
    }
    for key, value in DATASET_SOURCES.items()
])
display(sources_table)


,dataset_key,dataset_name,target_column,source_url
0,general_disease_symptoms,General Disease-Symptom Dataset,Disease,https://raw.githubusercontent.com/vyasanbmathe...
1,heart_disease,Heart Disease Dataset,Heart Disease Status,https://raw.githubusercontent.com/vyasanbmathe...
2,diabetes,Diabetes Dataset,Target,https://raw.githubusercontent.com/vyasanbmathe...


## 4. Loading Multiple Datasets

In [3]:
def standardize_column_name(column):
    """Convert a column name into a consistent snake_case format."""
    column = str(column).strip()
    column = re.sub(r"[^A-Za-z0-9]+", "_", column)
    column = re.sub(r"_+", "_", column).strip("_").lower()
    return column

def load_csv_from_url(url, dataset_name, timeout=30):
    """Load a CSV from a URL without crashing the whole notebook if unavailable."""
    try:
        response = requests.get(url, timeout=timeout, headers={"User-Agent": "Mozilla/5.0"})
        response.raise_for_status()
        try:
            df = pd.read_csv(io.BytesIO(response.content))
        except pd.errors.ParserError:
            # Some public CSV exports contain a small number of malformed rows.
            # The Python parser can continue while reporting skipped rows; this
            # keeps exploratory inspection available without changing the source.
            print(f"Parser warning for {dataset_name}; retrying with the Python parser.")
            df = pd.read_csv(
                io.BytesIO(response.content),
                engine="python",
                on_bad_lines="warn"
            )
        print(f"Loaded: {dataset_name}")
        print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
        return df
    except Exception as error:
        print(f"Could not load {dataset_name}: {error}")
        return None

raw_datasets = {}
for key, info in DATASET_SOURCES.items():
    raw_datasets[key] = load_csv_from_url(info["url"], info["name"])

available_datasets = {
    key: df for key, df in raw_datasets.items() if df is not None
}
print("\nAvailable datasets:", list(available_datasets.keys()))


Parser warning for General Disease-Symptom Dataset; retrying with the Python parser.
Loaded: General Disease-Symptom Dataset
Rows: 11571, Columns: 13
Loaded: Heart Disease Dataset
Rows: 10000, Columns: 21


Loaded: Diabetes Dataset
Rows: 70000, Columns: 34

Available datasets: ['general_disease_symptoms', 'heart_disease', 'diabetes']


## 5. Displaying Sample Data

In [4]:
for key, df in available_datasets.items():
    print("=" * 90)
    print(DATASET_SOURCES[key]["name"])
    print("Source:", DATASET_SOURCES[key]["url"])
    display(df.head(10))


General Disease-Symptom Dataset
Source: https://raw.githubusercontent.com/vyasanbmathew2008/Team-6/refs/heads/main/healthcare-disease-prediction/dataset/healthcare_dataset.csv


,Disease,Symptom_1,Symptom_2,Symptom_3,Symptom_4,Symptom_5,Symptom_6,Symptom_7,Symptom_8,Symptom_9,Symptom_10,Symptom_11,Symptom_12
0,Roseola,Runny nose,Rash,Red eyes,Loss of appetite,Rashes,Feverish,NaN,NaN,NaN,NaN,NaN,NaN
1,Roseola,High fever,Runny nose,Irritability,Headache,Cough,Rashes,Chills,NaN,NaN,NaN,NaN,NaN
2,Norovirus Infection,Stomach cramps,Nausea,Fatigue,Abdominal pain,Loss of appetite,Dehydration,Chills,Sweating,Rashes,General discomfort,Fever,NaN
3,Roseola,High fever,Runny nose,Irritability,Fatigue,Red eyes,Diarrhea,Vomiting,Feverish,NaN,NaN,NaN,NaN
4,Norovirus Infection,Diarrhea,Low-grade fever,Nausea,Headache,Fatigue,Loss of appetite,Dehydration,Chills,Sweating,General discomfort,Irritability,NaN
5,Shigellosis (Bacillary Dysentery),Abdominal pain or cramps,Diarrhea,Nausea,Vomiting,Headache,Fatigue,Dehydration,Muscle aches,Body aches,Rashes,Joint pain,NaN
6,Norovirus Infection,Diarrhea,Vomiting,Nausea,Headache,Fatigue,Abdominal pain,Loss of appetite,Sweating,Body aches,Fever,NaN,NaN
7,Shigellosis (Bacillary Dysentery),Fever,Abdominal pain or cramps,Diarrhea,Nausea,Fatigue,Dehydration,Loss of appetite,Muscle aches,Rashes,NaN,NaN,NaN
8,Shigellosis (Bacillary Dysentery),Fever,Diarrhea,Nausea,Loss of appetite,Feverish,Rashes,General discomfort,Chills,Joint pain,NaN,NaN,NaN
9,Roseola,High fever,Rash,Sore throat,Cough,Loss of appetite,Abdominal pain,Body aches,Chills,Feverish,NaN,NaN,NaN


Heart Disease Dataset
Source: https://raw.githubusercontent.com/vyasanbmathew2008/Team-6/refs/heads/main/heart_disease.csv


,Age,Gender,Blood Pressure,Cholesterol Level,Exercise Habits,Smoking,Family Heart Disease,Diabetes,BMI,High Blood Pressure,...,High LDL Cholesterol,Alcohol Consumption,Stress Level,Sleep Hours,Sugar Consumption,Triglyceride Level,Fasting Blood Sugar,CRP Level,Homocysteine Level,Heart Disease Status
0,56.0,Male,153.0,155.0,High,Yes,Yes,No,24.991591,Yes,...,No,High,Medium,7.633228,Medium,342.0,NaN,12.969246,12.387250,No
1,69.0,Female,146.0,286.0,High,No,Yes,Yes,25.221799,No,...,No,Medium,High,8.744034,Medium,133.0,157.0,9.355389,19.298875,No
2,46.0,Male,126.0,216.0,Low,No,No,No,29.855447,No,...,Yes,Low,Low,4.440440,Low,393.0,92.0,12.709873,11.230926,No
3,32.0,Female,122.0,293.0,High,Yes,Yes,No,24.130477,Yes,...,Yes,Low,High,5.249405,High,293.0,94.0,12.509046,5.961958,No
4,60.0,Male,166.0,242.0,Low,Yes,Yes,Yes,20.486289,Yes,...,No,Low,High,7.030971,High,263.0,154.0,10.381259,8.153887,No
5,25.0,Male,152.0,257.0,Low,Yes,No,No,28.144681,No,...,No,Low,Medium,5.504876,Low,126.0,91.0,4.297575,10.815983,No
6,78.0,Female,121.0,175.0,High,Yes,Yes,Yes,18.042332,No,...,No,Medium,Medium,9.240911,Medium,107.0,85.0,11.582983,19.659461,No
7,38.0,Female,161.0,187.0,Low,Yes,Yes,Yes,34.736683,No,...,No,Low,Medium,7.841008,High,228.0,111.0,4.929381,17.146599,No
8,56.0,Female,135.0,291.0,Low,No,Yes,Yes,34.493112,Yes,...,Yes,High,Low,6.941403,High,317.0,103.0,5.119015,6.051129,No
9,75.0,Male,144.0,252.0,Low,Yes,Yes,No,30.142149,No,...,Yes,Low,Medium,4.002662,High,199.0,96.0,10.005698,7.604357,No


Diabetes Dataset
Source: https://raw.githubusercontent.com/vyasanbmathew2008/Team-6/refs/heads/main/diabetes_dataset.csv


,Target,Genetic Markers,Autoantibodies,Family History,Environmental Factors,Insulin Levels,Age,BMI,Physical Activity,Dietary Habits,...,Pulmonary Function,Cystic Fibrosis Diagnosis,Steroid Use History,Genetic Testing,Neurological Assessments,Liver Function Tests,Digestive Enzyme Levels,Urine Test,Birth Weight,Early Onset Symptoms
0,Steroid-Induced Diabetes,Positive,Negative,No,Present,40,44,38,High,Healthy,...,76,No,No,Positive,3,Normal,56,Ketones Present,2629,No
1,Neonatal Diabetes Mellitus (NDM),Positive,Negative,No,Present,13,1,17,High,Healthy,...,60,Yes,No,Negative,1,Normal,28,Glucose Present,1881,Yes
2,Prediabetic,Positive,Positive,Yes,Present,27,36,24,High,Unhealthy,...,80,Yes,No,Negative,1,Abnormal,55,Ketones Present,3622,Yes
3,Type 1 Diabetes,Negative,Positive,No,Present,8,7,16,Low,Unhealthy,...,89,Yes,No,Positive,2,Abnormal,60,Ketones Present,3542,No
4,Wolfram Syndrome,Negative,Negative,Yes,Present,17,10,17,High,Healthy,...,41,No,No,Positive,1,Normal,24,Protein Present,1770,No
5,LADA,Positive,Negative,Yes,Present,17,41,26,Moderate,Healthy,...,85,Yes,No,Negative,2,Normal,52,Ketones Present,3835,Yes
6,Type 2 Diabetes,Negative,Negative,No,Absent,29,30,31,Moderate,Healthy,...,64,Yes,Yes,Negative,3,Abnormal,96,Ketones Present,4426,No
7,Wolcott-Rallison Syndrome,Positive,Negative,No,Absent,10,3,18,Low,Unhealthy,...,44,Yes,No,Negative,1,Normal,29,Ketones Present,1644,Yes
8,Secondary Diabetes,Negative,Positive,No,Absent,47,47,25,High,Healthy,...,71,No,Yes,Positive,3,Normal,74,Ketones Present,3721,No
9,Secondary Diabetes,Positive,Negative,Yes,Present,21,72,24,Low,Unhealthy,...,69,Yes,Yes,Positive,2,Abnormal,42,Protein Present,4206,No


## 6. Dataset Exploration

In [5]:
for key, df in available_datasets.items():
    print("=" * 90)
    print(DATASET_SOURCES[key]["name"])
    print("Shape:", df.shape)
    print("Column names:")
    print(list(df.columns))
    print("\nData types:")
    display(df.dtypes.to_frame("dtype"))
    print("\nInfo:")
    df.info()
    print("\nDescriptive summary:")
    display(df.describe(include="all").transpose().head(50))
    print("\nMissing values:")
    display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))


General Disease-Symptom Dataset
Shape: (11571, 13)
Column names:
['Disease', 'Symptom_1', 'Symptom_2', 'Symptom_3', 'Symptom_4', 'Symptom_5', 'Symptom_6', 'Symptom_7', 'Symptom_8', 'Symptom_9', 'Symptom_10', 'Symptom_11', 'Symptom_12']

Data types:


,dtype
Disease,str
Symptom_1,str
Symptom_2,str
Symptom_3,str
Symptom_4,str
Symptom_5,str
Symptom_6,str
Symptom_7,str
Symptom_8,str
Symptom_9,str



Info:
<class 'pandas.DataFrame'>
RangeIndex: 11571 entries, 0 to 11570
Data columns (total 13 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Disease     11571 non-null  str  
 1   Symptom_1   11571 non-null  str  
 2   Symptom_2   11571 non-null  str  
 3   Symptom_3   11571 non-null  str  
 4   Symptom_4   11571 non-null  str  
 5   Symptom_5   11346 non-null  str  
 6   Symptom_6   10837 non-null  str  
 7   Symptom_7   9834 non-null   str  
 8   Symptom_8   8216 non-null   str  
 9   Symptom_9   6179 non-null   str  
 10  Symptom_10  4030 non-null   str  
 11  Symptom_11  2119 non-null   str  
 12  Symptom_12  792 non-null    str  
dtypes: str(13)
memory usage: 1.1 MB

Descriptive summary:


,count,unique,top,freq
Disease,11571,16,Norovirus Infection,3372
Symptom_1,11571,64,Fever,2119
Symptom_2,11571,83,Vomiting,1193
Symptom_3,11571,81,Fatigue,1155
Symptom_4,11571,85,Headache,1505
Symptom_5,11346,73,Headache,1335
Symptom_6,10837,67,Loss of appetite,1204
Symptom_7,9834,57,Body aches,1113
Symptom_8,8216,41,Rashes,1092
Symptom_9,6179,32,Rashes,956



Missing values:


...


Info:


<class 'pandas.DataFrame'>
RangeIndex: 70000 entries, 0 to 69999
Data columns (total 34 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   Target                         70000 non-null  str  
 1   Genetic Markers                70000 non-null  str  
 2   Autoantibodies                 70000 non-null  str  
 3   Family History                 70000 non-null  str  
 4   Environmental Factors          70000 non-null  str  
 5   Insulin Levels                 70000 non-null  int64
 6   Age                            70000 non-null  int64
 7   BMI                            70000 non-null  int64
 8   Physical Activity              70000 non-null  str  
 9   Dietary Habits                 70000 non-null  str  
 10  Blood Pressure                 70000 non-null  int64
 11  Cholesterol Levels             70000 non-null  int64
 12  Waist Circumference            70000 non-null  int64
 13  Blood Glucose Levels       

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Target,70000,13,MODY,5553,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Genetic Markers,70000,2,Positive,35101,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Autoantibodies,70000,2,Negative,35058,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Family History,70000,2,Yes,35168,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Environmental Factors,70000,2,Absent,35088,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Insulin Levels,70000.0,NaN,NaN,NaN,21.607443,10.785852,5.0,13.0,19.0,28.0,49.0
Age,70000.0,NaN,NaN,NaN,32.0207,21.043173,0.0,14.0,31.0,49.0,79.0
BMI,70000.0,NaN,NaN,NaN,24.782943,6.014236,12.0,20.0,25.0,29.0,39.0
Physical Activity,70000,3,Moderate,23427,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Dietary Habits,70000,2,Healthy,35020,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Missing values:


,missing_count
Target,0
Genetic Markers,0
Autoantibodies,0
Family History,0
Environmental Factors,0
Insulin Levels,0
Age,0
BMI,0
Physical Activity,0
Dietary Habits,0


## 7. Data Cleaning

In [6]:
def clean_basic_dataframe(df):
    """Perform safe, general cleaning without assuming medical meaning."""
    cleaned = df.copy()

    # Standardize column names.
    cleaned.columns = [standardize_column_name(c) for c in cleaned.columns]

    # Treat common textual missing-value markers as actual missing values.
    missing_markers = ["", " ", "na", "n/a", "null", "none", "missing", "?"]
    cleaned = cleaned.replace(missing_markers, np.nan)

    # Remove exact duplicate rows.
    before = len(cleaned)
    cleaned = cleaned.drop_duplicates().reset_index(drop=True)
    print(f"Removed {before - len(cleaned)} duplicate rows.")

    # Strip whitespace from object/string columns.
    for col in cleaned.select_dtypes(include=["object", "string"]).columns:
        cleaned[col] = cleaned[col].astype("object").apply(lambda value: value.strip() if isinstance(value, str) else value).where(cleaned[col].notna(), np.nan)

    return cleaned

cleaned_datasets = {}
for key, df in available_datasets.items():
    print("\nCleaning:", DATASET_SOURCES[key]["name"])
    cleaned_datasets[key] = clean_basic_dataframe(df)
    print("Cleaned shape:", cleaned_datasets[key].shape)



Cleaning: General Disease-Symptom Dataset
Removed 0 duplicate rows.
Cleaned shape: (11571, 13)

Cleaning: Heart Disease Dataset


Removed 0 duplicate rows.
Cleaned shape: (10000, 21)

Cleaning: Diabetes Dataset


Removed 0 duplicate rows.


Cleaned shape: (70000, 34)


## 8. Data Preprocessing and Compatibility Checks

In [7]:
def find_standardized_target(df, original_target):
    """Find a target column after column-name standardization."""
    target = standardize_column_name(original_target)
    if target in df.columns:
        return target
    return None

for key, df in cleaned_datasets.items():
    target = find_standardized_target(df, DATASET_SOURCES[key]["target"])
    print(f"{key}: expected target={standardize_column_name(DATASET_SOURCES[key]['target'])}, found={target}")
    if target is not None:
        print("Target value counts:")
        display(df[target].value_counts(dropna=False).head(20).to_frame("count"))

print("\nCompatibility note:")
print(
    "The datasets have different targets, feature meanings, units, and label spaces. "
    "They must not be concatenated row-wise or merged by column name without a documented "
    "data-integration design. This notebook therefore analyzes them separately."
)


general_disease_symptoms: expected target=disease, found=disease
Target value counts:


,count
disease,
Norovirus Infection,3372
Shigellosis (Bacillary Dysentery),3339
Roseola,3269
"Hand, Foot, and Mouth Disease",881
Mumps,228
Scarlet Fever,190
Pertussis (Whooping Cough),98
Fifth Disease (Erythema Infectiosum),46
Common Cold,44


heart_disease: expected target=heart_disease_status, found=heart_disease_status
Target value counts:


,count
heart_disease_status,
No,8000
Yes,2000


diabetes: expected target=target, found=target
Target value counts:


,count
target,
MODY,5553
Secondary Diabetes,5479
Cystic Fibrosis-Related Diabetes (CFRD),5464
Type 1 Diabetes,5446
Neonatal Diabetes Mellitus (NDM),5408
Wolcott-Rallison Syndrome,5400
Type 2 Diabetes,5397
Prediabetic,5376
Gestational Diabetes,5344



Compatibility note:
The datasets have different targets, feature meanings, units, and label spaces. They must not be concatenated row-wise or merged by column name without a documented data-integration design. This notebook therefore analyzes them separately.


## 9. Sample Data Demonstration

The following rows are illustrative examples only. They are not genuine medical records and must not be treated as clinical evidence.

In [8]:
sample_patient_format = pd.DataFrame([
    [45, "Male", "Yes", "Yes", "Yes", "No", "Yes", 1],
    [28, "Female", "No", "No", "No", "No", "No", 0],
    [62, "Male", "Yes", "Yes", "Yes", "Yes", "Yes", 1],
    [35, "Female", "No", "Yes", "No", "No", "Yes", 0],
    [51, "Male", "Yes", "Yes", "Yes", "No", "Yes", 1],
], columns=[
    "Age", "Gender", "Smoking", "Cough", "Breathlessness",
    "Chest_Pain", "Fatigue", "Lung_Disease"
])
display(sample_patient_format)
print("These values are synthetic illustrations, not patient records.")


,Age,Gender,Smoking,Cough,Breathlessness,Chest_Pain,Fatigue,Lung_Disease
0,45,Male,Yes,Yes,Yes,No,Yes,1
1,28,Female,No,No,No,No,No,0
2,62,Male,Yes,Yes,Yes,Yes,Yes,1
3,35,Female,No,Yes,No,No,Yes,0
4,51,Male,Yes,Yes,Yes,No,Yes,1


These values are synthetic illustrations, not patient records.


## 10. Selecting a Demonstration Task and Train-Test Splitting

Because the supplied files do not include a verified lung-disease target, the default task below uses `Heart Disease Status` if it is present and binary. The exported file is consequently a heart-disease demonstration model, not a lung-disease model. To train a true lung-disease model, replace the dataset URL and mapping with a verified lung-disease dataset.

In [9]:
# Select a safe, available binary target.
TASK_KEY = None
TARGET_COLUMN = None

preferred_tasks = [
    ("heart_disease", "heart_disease_status"),
]

for key, target in preferred_tasks:
    if key in cleaned_datasets and target in cleaned_datasets[key].columns:
        unique_count = cleaned_datasets[key][target].dropna().nunique()
        if unique_count == 2:
            TASK_KEY = key
            TARGET_COLUMN = target
            break

if TASK_KEY is None:
    raise ValueError(
        "No suitable binary demonstration target was found. "
        "Add a verified binary dataset and update the task-selection section."
    )

task_df = cleaned_datasets[TASK_KEY].copy()
task_df = task_df.dropna(subset=[TARGET_COLUMN]).reset_index(drop=True)

X = task_df.drop(columns=[TARGET_COLUMN])
y_raw = task_df[TARGET_COLUMN].astype(str).str.strip()

# Encode the target only; feature preprocessing remains inside the pipeline.
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y_raw)

print("Selected dataset:", DATASET_SOURCES[TASK_KEY]["name"])
print("Target column:", TARGET_COLUMN)
print("Target classes:", list(target_encoder.classes_))
print("Feature shape:", X.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


Selected dataset: Heart Disease Dataset
Target column: heart_disease_status
Target classes: ['No', 'Yes']
Feature shape: (10000, 20)
Training rows: 8000
Testing rows: 2000


## 11. Exploratory Data Analysis: Histograms and Boxplots

The following plots describe the selected modeling dataset before preprocessing. Histograms show the distribution of numeric variables, while boxplots highlight spread and potential outliers. These are exploratory views only; an apparent outlier is not automatically a data error or a medical abnormality.

In [10]:
# Target balance
plt.figure(figsize=(7, 4))
target_counts = y_raw.value_counts(dropna=False).rename_axis(TARGET_COLUMN).reset_index(name="count")
sns.barplot(data=target_counts, x=TARGET_COLUMN, y="count", hue=TARGET_COLUMN, legend=False, palette="Set2")
plt.title(f"Target class distribution: {TARGET_COLUMN}")
plt.xlabel("Class")
plt.ylabel("Rows")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

# Numeric feature distributions
eda_numeric = X.select_dtypes(include=["number", "bool"]).copy()
if eda_numeric.shape[1] == 0:
    print("No numeric features available for histograms or boxplots.")
else:
    n_features = eda_numeric.shape[1]
    n_cols = 3
    n_rows = int(np.ceil(n_features / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
    axes = np.atleast_1d(axes).ravel()
    for ax, column in zip(axes, eda_numeric.columns):
        sns.histplot(eda_numeric[column].dropna(), kde=True, ax=ax, color="#2a9d8f")
        ax.set_title(f"Histogram: {column}")
        ax.set_xlabel(column)
    for ax in axes[n_features:]:
        ax.remove()
    fig.suptitle("Numeric Feature Histograms", fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(max(10, 1.2 * n_features), 6))
    sns.boxplot(data=eda_numeric, orient="h", color="#e9c46a")
    plt.title("Numeric Feature Boxplots")
    plt.xlabel("Value")
    plt.tight_layout()
    plt.show()

    display(eda_numeric.describe().T.assign( missing=eda_numeric.isna().sum(), outlier_iqr_count=[
        ((eda_numeric[c] < (eda_numeric[c].quantile(0.25) - 1.5 * (eda_numeric[c].quantile(0.75) - eda_numeric[c].quantile(0.25)))) |
         (eda_numeric[c] > (eda_numeric[c].quantile(0.75) + 1.5 * (eda_numeric[c].quantile(0.75) - eda_numeric[c].quantile(0.25))))).sum()
        for c in eda_numeric.columns
    ]))


,count,mean,std,min,25%,50%,75%,max,missing,outlier_iqr_count
age,9971.0,49.296259,18.193970,18.000000,34.000000,49.000000,65.000000,80.000000,29,0
blood_pressure,9981.0,149.757740,17.572969,120.000000,134.000000,150.000000,165.000000,180.000000,19,0
cholesterol_level,9970.0,225.425577,43.575809,150.000000,187.000000,226.000000,263.000000,300.000000,30,0
bmi,9978.0,29.077269,6.307098,18.002837,23.658075,29.079492,34.520015,39.996954,22,0
sleep_hours,9975.0,6.991329,1.753195,4.000605,5.449866,7.003252,8.531577,9.999952,25,0
triglyceride_level,9974.0,250.734409,87.067226,100.000000,176.000000,250.000000,326.000000,400.000000,26,0
fasting_blood_sugar,9978.0,120.142213,23.584011,80.000000,99.000000,120.000000,141.000000,160.000000,22,0
crp_level,9974.0,7.472201,4.340248,0.003647,3.674126,7.472164,11.255592,14.997087,26,0
homocysteine_level,9980.0,12.456271,4.323426,5.000236,8.723334,12.409395,16.140564,19.999037,20,0


## 11. Building the Preprocessing Pipeline

In [11]:
# Identify numerical and categorical columns from the training data only.
numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = [c for c in X_train.columns if c not in numeric_features]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ],
    remainder="drop"
)

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)


Numerical features: ['age', 'blood_pressure', 'cholesterol_level', 'bmi', 'sleep_hours', 'triglyceride_level', 'fasting_blood_sugar', 'crp_level', 'homocysteine_level']
Categorical features: ['gender', 'exercise_habits', 'smoking', 'family_heart_disease', 'diabetes', 'high_blood_pressure', 'low_hdl_cholesterol', 'high_ldl_cholesterol', 'alcohol_consumption', 'stress_level', 'sugar_consumption']


## 12. Model Training

In [12]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=8, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=250, class_weight="balanced", random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "Support Vector Machine": SVC(
        probability=True, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        random_state=RANDOM_STATE
    )
}

fitted_pipelines = {}
predictions = {}
evaluation_rows = {}

for model_name, model in models.items():
    print(f"Training {model_name}...")
    pipeline = Pipeline(steps=[
        ("preprocessing", preprocessor),
        ("model", model)
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    fitted_pipelines[model_name] = pipeline
    predictions[model_name] = y_pred

    evaluation_rows[model_name] = {
        "model": model_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "f1_score": f1_score(y_test, y_pred, average="weighted", zero_division=0)
    }

print("Training complete.")


Training Logistic Regression...
Training Decision Tree...


Training Random Forest...


Training Support Vector Machine...


Training Gradient Boosting...


Training complete.


## 13. Model Evaluation

In [13]:
for model_name, y_pred in predictions.items():
    print("=" * 90)
    print(model_name)
    print("\nClassification report:")
    print(classification_report(
        y_test, y_pred,
        target_names=target_encoder.classes_,
        zero_division=0
    ))

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=target_encoder.classes_,
        yticklabels=target_encoder.classes_
    )
    plt.title(f"Confusion Matrix - {model_name}")
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.tight_layout()
    plt.show()


Logistic Regression

Classification report:
              precision    recall  f1-score   support

          No       0.79      0.51      0.62      1600
         Yes       0.19      0.47      0.27       400

    accuracy                           0.50      2000
   macro avg       0.49      0.49      0.45      2000
weighted avg       0.67      0.50      0.55      2000

Decision Tree

Classification report:
              precision    recall  f1-score   support

          No       0.80      0.55      0.65      1600
         Yes       0.20      0.44      0.27       400

    accuracy                           0.53      2000
   macro avg       0.50      0.49      0.46      2000
weighted avg       0.68      0.53      0.58      2000



Random Forest

Classification report:
              precision    recall  f1-score   support

          No       0.80      1.00      0.89      1600
         Yes       0.00      0.00      0.00       400

    accuracy                           0.80      2000
   macro avg       0.40      0.50      0.44      2000
weighted avg       0.64      0.80      0.71      2000

Support Vector Machine

Classification report:
              precision    recall  f1-score   support

          No       0.80      0.60      0.68      1600
         Yes       0.20      0.40      0.27       400

    accuracy                           0.56      2000
   macro avg       0.50      0.50      0.48      2000
weighted avg       0.68      0.56      0.60      2000



Gradient Boosting

Classification report:
              precision    recall  f1-score   support

          No       0.80      1.00      0.89      1600
         Yes       1.00      0.00      0.00       400

    accuracy                           0.80      2000
   macro avg       0.90      0.50      0.45      2000
weighted avg       0.84      0.80      0.71      2000



## 14. Model Comparison

Accuracy alone is not sufficient for healthcare-related classification. Recall measures how many actual positive cases were detected. A false negative occurs when a positive case is incorrectly predicted as negative; in a screening context, this can be especially important. Precision, recall, F1-score, class-specific metrics, calibration, and the consequences of errors should all be considered.

In [14]:
comparison_df = pd.DataFrame(list(evaluation_rows.values()))
comparison_df = comparison_df.sort_values(
    by=["recall", "f1_score"], ascending=False
).reset_index(drop=True)

display(comparison_df)

print(
    "The table contains actual results produced on this train/test split. "
    "Do not treat these metrics as clinical validation."
)


,model,accuracy,precision,recall,f1_score
0,Gradient Boosting,0.8005,0.840320,0.8005,0.712306
1,Random Forest,0.7980,0.639679,0.7980,0.710122
2,Support Vector Machine,0.5580,0.679816,0.5580,0.600286
3,Decision Tree,0.5285,0.676592,0.5285,0.575453
4,Logistic Regression,0.5005,0.671936,0.5005,0.550273


The table contains actual results produced on this train/test split. Do not treat these metrics as clinical validation.


## 15. Final Model Selection

In [15]:
# Educational selection rule:
# prioritize weighted recall, then weighted F1-score.
# This is not a clinical decision rule.
selected_model_name = comparison_df.iloc[0]["model"]
final_pipeline = fitted_pipelines[selected_model_name]

print("Selected model for export:", selected_model_name)
print(
    "Selection was based on the displayed test-set metrics, prioritizing recall and then F1-score. "
    "For real healthcare deployment, use external validation, subgroup analysis, calibration, "
    "and domain-expert review."
)


Selected model for export: Gradient Boosting
Selection was based on the displayed test-set metrics, prioritizing recall and then F1-score. For real healthcare deployment, use external validation, subgroup analysis, calibration, and domain-expert review.


## 16. Saving the Complete Pipeline

In [16]:
export_bundle = {
    "pipeline": final_pipeline,
    "target_encoder": target_encoder,
    "feature_columns": list(X.columns),
    "dataset_key": TASK_KEY,
    "target_column": TARGET_COLUMN,
    "selected_model_name": selected_model_name,
    "random_state": RANDOM_STATE,
    "note": (
        "Educational model only. Not a medical diagnosis system. "
        "The pipeline includes imputing, encoding, scaling, and the classifier."
    )
}

joblib.dump(export_bundle, MODEL_FILE)
print(f"Saved model bundle to: {MODEL_FILE}")
print("File exists:", os.path.exists(MODEL_FILE))


Saved model bundle to: disease_model.pkl
File exists: True


## 17. Loading and Testing the Saved Model

In [17]:
loaded_bundle = joblib.load(MODEL_FILE)
loaded_pipeline = loaded_bundle["pipeline"]
loaded_target_encoder = loaded_bundle["target_encoder"]
expected_features = loaded_bundle["feature_columns"]

# Create a sample record using the first test row so that all required
# feature columns are available. This is only a software test fixture.
sample_record = X_test.iloc[[0]].copy()

loaded_encoded_prediction = loaded_pipeline.predict(sample_record)
loaded_label_prediction = loaded_target_encoder.inverse_transform(
    loaded_encoded_prediction.astype(int)
)

print("Sample input record:")
display(sample_record)
print("Predicted class:", loaded_label_prediction[0])

if hasattr(loaded_pipeline, "predict_proba"):
    probabilities = loaded_pipeline.predict_proba(sample_record)[0]
    probability_table = pd.DataFrame({
        "class": loaded_target_encoder.classes_,
        "probability": probabilities
    })
    display(probability_table)
else:
    print("Probability output is not available for this pipeline.")


Sample input record:


,age,gender,blood_pressure,cholesterol_level,exercise_habits,smoking,family_heart_disease,diabetes,bmi,high_blood_pressure,low_hdl_cholesterol,high_ldl_cholesterol,alcohol_consumption,stress_level,sleep_hours,sugar_consumption,triglyceride_level,fasting_blood_sugar,crp_level,homocysteine_level
6177,63.0,Male,130.0,234.0,High,Yes,No,Yes,38.078,Yes,No,Yes,Low,Low,8.860326,High,284.0,120.0,6.762998,15.550978


Predicted class: No


,class,probability
0,No,0.810525
1,Yes,0.189475


## 18. Limitations and Ethical Considerations

- The supplied datasets may be synthetic, simulated, scraped, or otherwise non-clinical. Verify each original source before describing provenance.
- The default demonstration is based on the supplied heart-disease file, not a verified lung-disease dataset.
- A random train/test split is not a substitute for external validation.
- Dataset imbalance, duplicate patterns, label noise, and hidden leakage may distort metrics.
- Accuracy, precision, recall, and F1-score do not establish clinical safety.
- False negatives and false positives have different consequences and should be assessed with domain experts.
- The model may perform differently across demographic or socioeconomic groups.
- Do not use the exported file to diagnose, triage, prescribe, or recommend treatment.
- A future system should include external validation, calibration, uncertainty handling, human review, audit logs, privacy protections, and clear referral guidance.


## 19. Final Conclusion

This notebook creates a reproducible end-to-end machine-learning pipeline, evaluates multiple classifiers, displays real metrics generated from the available data, and exports the complete preprocessing-and-model pipeline as `disease_model.pkl`.

To convert this demonstration into a genuine lung-disease project, add a verified lung-disease dataset with a clearly defined target, update `DATASET_SOURCES`, confirm the feature meanings and label semantics, and rerun the notebook. Do not merge unrelated disease datasets merely because they are all healthcare-related.